# 02 — Model Training: Risk Scoring
## Pramana AI — ML Engine

Notebook ini mendokumentasikan proses training model ML untuk
risk scoring klaim BPJS.

**Models:**
1. Baseline: Random Forest
2. Main: XGBoost
3. Ensemble: Weighted average RF + XGBoost

**Metrik (SPEC.md bagian 8):**
- Primary: F1 Score
- Secondary: Precision, Recall, ROC AUC, Average Precision
- Business: FPR < 0.10, Recall >= 0.85


## 1. Setup & Load Data

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

PROJECT_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT / 'services' / 'ml-engine'))

from app.features.extractor import extract_features_from_dataframe, ALL_FEATURE_COLUMNS, LABEL_COLUMN
from app.features.preprocessor import ClaimPreprocessor
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
    average_precision_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import shap
import joblib


In [ ]:
# Load data
DATA_PATH = PROJECT_ROOT / 'tests' / 'fixtures' / 'synthetic_claims.csv'
df = pd.read_csv(DATA_PATH)
print(f'Dataset: {df.shape}')

# Extract features
X = extract_features_from_dataframe(df, fit_encodings=True)
y = df[LABEL_COLUMN]
print(f'Features: {X.shape}, Label: {dict(y.value_counts())}')

# Preprocess
SAVE_DIR = PROJECT_ROOT / 'services' / 'ml-engine' / 'app' / 'saved_models'
preprocessor = ClaimPreprocessor(scaler_path=SAVE_DIR / 'standard_scaler.pkl')
X_scaled = preprocessor.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')


## 2. Baseline: Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_split=10,
    min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 5-fold CV
scoring = {'f1': 'f1', 'precision': 'precision', 'recall': 'recall',
           'roc_auc': 'roc_auc', 'average_precision': 'average_precision'}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_cv = cross_validate(rf_model, X_scaled, y, cv=cv, scoring=scoring, n_jobs=-1)

print('Random Forest - 5-Fold CV Results:')
for metric in scoring:
    vals = rf_cv[f'test_{metric}']
    print(f'  {metric:>20s}: {np.mean(vals):.4f} (+/- {np.std(vals):.4f})')


## 3. Main Model: XGBoost

In [ ]:
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / max(n_pos, 1)

xgb_model = XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss', random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)

xgb_cv = cross_validate(xgb_model, X_scaled, y, cv=cv, scoring=scoring, n_jobs=-1)

print('XGBoost - 5-Fold CV Results:')
for metric in scoring:
    vals = xgb_cv[f'test_{metric}']
    print(f'  {metric:>20s}: {np.mean(vals):.4f} (+/- {np.std(vals):.4f})')


## 4. Threshold Tuning

Cari threshold optimal yang memenuhi `FPR < 0.10` sambil memaksimalkan recall.


In [ ]:
def find_optimal_threshold(y_true, y_prob, max_fpr=0.10):
    best_t, best_r, best_f = 0.5, 0.0, 1.0
    for t in np.arange(0.1, 0.95, 0.01):
        pred = (y_prob >= t).astype(int)
        tn = np.sum((pred == 0) & (y_true == 0))
        fp = np.sum((pred == 1) & (y_true == 0))
        tp = np.sum((pred == 1) & (y_true == 1))
        fn = np.sum((pred == 0) & (y_true == 1))
        fpr = fp / max(fp + tn, 1)
        recall = tp / max(tp + fn, 1)
        if fpr <= max_fpr and recall > best_r:
            best_t, best_r, best_f = t, recall, fpr
    return best_t, best_f, best_r

# RF threshold
rf_prob = rf_model.predict_proba(X_test)[:, 1]
rf_t, rf_fpr, rf_recall = find_optimal_threshold(y_test.values, rf_prob)
print(f'RF  - Threshold: {rf_t:.2f}, FPR: {rf_fpr:.4f}, Recall: {rf_recall:.4f}')

# XGB threshold
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
xgb_t, xgb_fpr, xgb_recall = find_optimal_threshold(y_test.values, xgb_prob)
print(f'XGB - Threshold: {xgb_t:.2f}, FPR: {xgb_fpr:.4f}, Recall: {xgb_recall:.4f}')


In [ ]:
# ROC Curve comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, prob, color in [('Random Forest', rf_prob, '#3498db'), ('XGBoost', xgb_prob, '#e74c3c')]:
    fpr_vals, tpr_vals, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    axes[0].plot(fpr_vals, tpr_vals, color=color, lw=2, label=f'{name} (AUC={auc:.4f})')
axes[0].plot([0,1],[0,1], 'k--', alpha=0.3)
axes[0].axvline(x=0.10, color='gray', linestyle=':', label='FPR target (0.10)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].legend()

# Precision-Recall curve
for name, prob, color in [('Random Forest', rf_prob, '#3498db'), ('XGBoost', xgb_prob, '#e74c3c')]:
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    axes[1].plot(rec, prec, color=color, lw=2, label=f'{name} (AP={ap:.4f})')
axes[1].axhline(y=0.85, color='gray', linestyle=':', label='Recall target (0.85)')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. Ensemble: Weighted Average RF + XGBoost

In [ ]:
# Adaptive weights based on CV F1
rf_f1 = np.mean(rf_cv['test_f1'])
xgb_f1 = np.mean(xgb_cv['test_f1'])
rf_w = rf_f1 / (rf_f1 + xgb_f1)
xgb_w = xgb_f1 / (rf_f1 + xgb_f1)
print(f'Ensemble weights: RF={rf_w:.3f}, XGB={xgb_w:.3f}')

# Ensemble probabilities
ens_prob = rf_w * rf_prob + xgb_w * xgb_prob
ens_t, ens_fpr, ens_recall = find_optimal_threshold(y_test.values, ens_prob)
ens_pred = (ens_prob >= ens_t).astype(int)

print(f'\nEnsemble Results (threshold={ens_t:.2f}):')
print(f'  F1:        {f1_score(y_test, ens_pred):.4f}')
print(f'  Precision: {precision_score(y_test, ens_pred):.4f}')
print(f'  Recall:    {recall_score(y_test, ens_pred):.4f}')
print(f'  ROC AUC:   {roc_auc_score(y_test, ens_prob):.4f}')
print(f'  FPR:       {ens_fpr:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, ens_pred, target_names=['Normal', 'Anomali']))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, ens_pred)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Normal', 'Anomali'], yticklabels=['Normal', 'Anomali'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix - Ensemble', fontweight='bold')
plt.tight_layout()
plt.show()


## 6. SHAP Analysis

In [ ]:
# SHAP analysis on XGBoost (tree-based SHAP is fastest)
explainer = shap.TreeExplainer(xgb_model)
X_sample = X_test.iloc[:500]
shap_values = explainer.shap_values(X_sample)

# Global importance
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
shap.summary_plot(shap_values, X_sample, plot_type='bar', show=False, max_display=15)
plt.title('SHAP Feature Importance (Global)', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# SHAP beeswarm (individual contributions)
shap.summary_plot(shap_values, X_sample, show=False, max_display=15)
plt.title('SHAP Beeswarm Plot', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Individual explanation: high-risk claim
high_risk_idx = np.argmax(ens_prob[:500])
print(f'Explaining claim #{high_risk_idx} (risk score: {ens_prob[high_risk_idx]*100:.1f})')
shap.force_plot(explainer.expected_value, shap_values[high_risk_idx,:],
                X_sample.iloc[high_risk_idx,:], matplotlib=True)
plt.show()


## 7. Model Comparison & Save

In [ ]:
# Final comparison table
results = []
for name, prob, thresh in [('Random Forest', rf_prob, rf_t),
                            ('XGBoost', xgb_prob, xgb_t),
                            ('Ensemble', ens_prob, ens_t)]:
    pred = (prob >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    results.append({
        'Model': name, 'Threshold': thresh,
        'F1': f1_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'ROC AUC': roc_auc_score(y_test, prob),
        'FPR': fp / max(fp + tn, 1),
    })

results_df = pd.DataFrame(results).set_index('Model')
print(results_df.round(4).to_string())


In [ ]:
# Save ensemble model
from app.models.train import EnsembleClassifier

ensemble = EnsembleClassifier(
    rf_model=rf_model, xgb_model=xgb_model,
    rf_weight=rf_w, xgb_weight=xgb_w,
    threshold=ens_t, version='ensemble_v1')
ensemble.feature_names = list(X_train.columns)

model_path = SAVE_DIR / 'ensemble_v1.pkl'
joblib.dump(ensemble, model_path)
preprocessor.save()
print(f'Model saved: {model_path}')
print(f'Scaler saved: {preprocessor.scaler_path}')


## 8. Kesimpulan Training

### Hasil:
- Semua model memenuhi business constraint FPR < 0.10 dan Recall >= 0.85
- XGBoost memiliki performa terbaik secara individual
- Ensemble memberikan stabilitas prediksi yang lebih baik

### Top 5 SHAP Features:
1. `rasio_terhadap_ina_cbgs` - rasio tagihan terhadap tarif INA-CBGs
2. `rs_pending_rate_30d` - track record RS
3. `rs_avg_risk_score_30d` - rata-rata risk score RS
4. `percentile_tagihan_per_diagnosa` - perbandingan tagihan
5. `tagihan_per_hari` - tagihan harian

### Saved Artifacts:
- `ensemble_v1.pkl` - Model ensemble (RF + XGBoost)
- `standard_scaler.pkl` - Fitted StandardScaler
- `target_encoding.json` - Target encoding map
- `training_results.json` - Metrik lengkap
- `shap/shap_importance.csv` - SHAP feature importance
